_Image Classification using ViT_

- Classify chest Xray images using ViT in two classes.
	1. Build a vision transformer model using the basic layers from the framework.
	1. Resize images to 224x224.
	1. Apply suitable patching mechanism to convert images into patches to input to the ViT.
	1. Use ‘Finding Labels’ column to classify images in 2 classes: ‘No finding’ vs ‘With findings’ (every label combined other than no finding)
	1. Split the dataset in 70:10:20 ratio for training, validation and testing and make sure that the class distribution is maintained in all the splits.
	1. Train the model using the processed dataset obtained after applying above steps.
	1. Use all suitable classification metrics to test the model performance on the test set.

- Dataset:
	- Chest Xray dataset: [xray_dataset.zip](https://drive.google.com/file/d/1yDXWuWrFOKs8KKi4wbKcZk_D_GP3M5uk/view?usp=sharing)

In [15]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [11]:
import os
os.listdir('/kaggle/input/xray-dataset/xray_dataset')

['images', 'chest_xray.csv']

In [19]:
src_dir='/kaggle/input/xray-dataset/xray_dataset'

df = pd.read_csv(src_dir+'/chest_xray.csv')

df['Target'] = df['Finding Labels'].apply(lambda x: 0 if x == 'No Finding' else 1)
df['Image Path'] = df['Image Index'].apply(lambda x: os.path.join(src_dir+'/images', x))
df = df[df['Image Path'].apply(os.path.exists)]

display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2906 entries, 0 to 2905
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Image Index                  2906 non-null   object 
 1   Finding Labels               2906 non-null   object 
 2   Follow-up #                  2906 non-null   int64  
 3   Patient ID                   2906 non-null   int64  
 4   Patient Age                  2906 non-null   object 
 5   Patient Gender               2906 non-null   object 
 6   View Position                2906 non-null   object 
 7   OriginalImageWidth           2906 non-null   int64  
 8   OriginalImageHeight          2906 non-null   int64  
 9   OriginalImagePixelSpacing_x  2906 non-null   float64
 10  OriginalImagePixelSpacing_y  2906 non-null   float64
 11  Target                       2906 non-null   int64  
 12  Image Path                   2906 non-null   object 
dtypes: float64(2), int

None

In [22]:
train_df, temp_df = train_test_split(
    df, 
    test_size=0.3, 
    stratify=df['Target'], 
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=2/3,
    stratify=temp_df['Target'], 
    random_state=42
)

display(len(train_df))
display(len(val_df))
display(len(test_df))

2034

290

582

In [27]:
IMG_SIZE = 224
BATCH_SIZE = 32

def preprocess_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.int32)

def make_dataset(df):
    paths = df['Image Path'].values
    labels = df['Target'].values
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)

display(train_dataset)
display(val_dataset)
display(test_dataset)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [31]:
class PatchExtractor(tf.keras.layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

class PatchEncoder(tf.keras.layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = tf.keras.layers.Dense(projection_dim)
        self.pos_embedding = self.add_weight(name="pos_embedding", shape=(1, num_patches, projection_dim))

    def call(self, patch):
        return self.projection(patch) + self.pos_embedding

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = tf.keras.layers.Dense(units, activation=tf.nn.gelu)(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    return x

def create_vit_classifier(input_shape=(224, 224, 3), patch_size=16, num_layers=4,num_heads=4, projection_dim=64, transformer_units=[128, 64], num_classes=1):

    inputs = tf.keras.Input(shape=input_shape)
    num_patches = (input_shape[0] // patch_size) ** 2

    x = PatchExtractor(patch_size)(inputs)
    x = PatchEncoder(num_patches, projection_dim)(x)

    for _ in range(num_layers):
        x1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = tf.keras.layers.Add()([x, attention_output])
        x3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        x = tf.keras.layers.Add()([x2, x3])

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [33]:
model = create_vit_classifier()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3             │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ patch_extractor_3         │ (None, None, 768)      │              0 │ input_layer_3[0][0]    │
│ (PatchExtractor)          │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ patch_encoder_3           │ (None, 196, 64)        │         61,760 │ patch_extractor_3[0][… │
│ (PatchEncoder)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_9     │ (None, 196, 64)        │            128 │ patch_encoder_3[0][0]  │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_4    │ (None, 196, 64)        │         66,368 │ layer_normalization_9… │
│ (MultiHeadAttention)      │                        │                │ layer_normalization_9… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_8 (Add)               │ (None, 196, 64)        │              0 │ patch_encoder_3[0][0], │
│                           │                        │                │ multi_head_attention_… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_10    │ (None, 196, 64)        │            128 │ add_8[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_13 (Dense)          │ (None, 196, 128)       │          8,320 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_14 (Dropout)      │ (None, 196, 128)       │              0 │ dense_13[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_14 (Dense)          │ (None, 196, 64)        │          8,256 │ dropout_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_15 (Dropout)      │ (None, 196, 64)        │              0 │ dense_14[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_9 (Add)               │ (None, 196, 64)        │              0 │ add_8[0][0],           │
│                           │                        │                │ dropout_15[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_11    │ (None, 196, 64)        │            128 │ add_9[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_5    │ (None, 196, 64)        │         66,368 │ layer_normalization_1… │
│ (MultiHeadAttention)      │                        │                │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_10 (Add)         

 Total params: 394,753 (1.51 MB)

 Trainable params: 394,753 (1.51 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.fit(train_dataset, validation_data=val_dataset, epochs=10)

Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.5208 - auc_1: 0.5016 - loss: 0.7207 - precision_1: 0.4400 - recall_1: 0.3488 - val_accuracy: 0.5448 - val_auc_1: 0.5589 - val_loss: 0.6871 - val_precision_1: 0.5204 - val_recall_1: 0.7286
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.5267 - auc_1: 0.5360 - loss: 0.6949 - precision_1: 0.4908 - recall_1: 0.3620 - val_accuracy: 0.5069 - val_auc_1: 0.5643 - val_loss: 0.6845 - val_precision_1: 0.4824 - val_recall_1: 0.2929
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - accuracy: 0.5268 - auc_1: 0.5069 - loss: 0.7006 - precision_1: 0.4341 - recall_1: 0.2973 - val_accuracy: 0.5552 - val_auc_1: 0.5633 - val_loss: 0.6862 - val_precision_1: 0.5311 - val_recall_1: 0.6714
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.5107 - auc_1: 0.5219 - loss: 0.7050 - precision_1: 0.4951 - recall_1: 0.4903 - val_accuracy: 0.5379 - val_auc_1: 0.5651 - val_loss: 0.6911 - val_precision_1: 0.6667 - val_recall_1:

In [38]:
y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_pred_probs = model.predict(test_dataset).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

19/19 ━━━━━━━━━━━━━━━━━━━━ 11s 347ms/step


In [39]:
print(classification_report(y_true, y_pred, target_names=["No Finding", "With Findings"]))
print("ROC AUC Score:", roc_auc_score(y_true, y_pred_probs))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

               precision    recall  f1-score   support

   No Finding       0.53      0.20      0.29       300
With Findings       0.49      0.81      0.61       282

     accuracy                           0.49       582
    macro avg       0.51      0.50      0.45       582
 weighted avg       0.51      0.49      0.44       582

ROC AUC Score: 0.5089479905437352
Confusion Matrix:
 [[ 59 241]
 [ 53 229]]
